# Anti-UAV YOLO26s — 2-Class: Bird vs Drone
**Classes:** Bird (confuser) | Drone (all UAVs merged)

**Expected results:** mAP@0.5 ~0.96-0.98 (easier 2-class problem)

**Before running:**
1. Upload `backup_merged_dataset_2class.tar.gz` to Drive
   (generated by running `python prepare_2class_run.py` locally)
2. Runtime → T4 GPU → Run all

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')
import subprocess
r = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
print('GPU:', r.stdout.strip())


In [ ]:
# Find and extract 2-class dataset
import tarfile, os, yaml

tar_path = None
for root, dirs, files in os.walk('/content/drive/MyDrive'):
    for f in files:
        if '2class' in f and f.endswith('.tar.gz'):
            tar_path = os.path.join(root, f)
            break
    if tar_path:
        break

if tar_path is None:
    raise FileNotFoundError('2-class dataset tar not found in Drive. Run prepare_2class_run.py first.')

print(f'Found: {tar_path}')
extract_dir = '/content/dataset'
os.makedirs(extract_dir, exist_ok=True)
print('Extracting...')
with tarfile.open(tar_path) as tf:
    tf.extractall(extract_dir)

# Find data.yaml
data_yaml = None
for root, dirs, files in os.walk(extract_dir):
    if 'data.yaml' in files:
        data_yaml = os.path.join(root, 'data.yaml')
        break

base = os.path.dirname(data_yaml)
with open(data_yaml) as f:
    cfg = yaml.safe_load(f)
cfg['path'] = base
cfg['train'] = os.path.join(base, 'train', 'images')
cfg['val'] = os.path.join(base, 'val', 'images')
cfg['test'] = os.path.join(base, 'test', 'images')
with open(data_yaml, 'w') as f:
    yaml.dump(cfg, f)

print(f'Classes: {cfg["names"]}')
for split in ['train', 'val', 'test']:
    print(f'  {split}: {len(os.listdir(cfg[split]))} images')


In [ ]:
# Install ultralytics
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'ultralytics>=8.4.0'], check=True)
import ultralytics
print(f'ultralytics {ultralytics.__version__}')


In [ ]:
# Train — Bird vs Drone 2-class
import threading, time, shutil, os
from ultralytics import YOLO

DRIVE_BACKUP = '/content/drive/MyDrive/anti_uav_2class_checkpoints'
os.makedirs(DRIVE_BACKUP, exist_ok=True)
stop_backup = threading.Event()

def backup_to_drive():
    while not stop_backup.is_set():
        time.sleep(600)
        for root, dirs, files in os.walk('/content/runs'):
            if 'weights' in dirs:
                weights_dir = os.path.join(root, 'weights')
                for f in os.listdir(weights_dir):
                    try:
                        shutil.copy2(os.path.join(weights_dir, f), os.path.join(DRIVE_BACKUP, f))
                        print(f'[backup] {f} → Drive')
                    except Exception as e:
                        print(f'[backup] Failed: {e}')
                break

threading.Thread(target=backup_to_drive, daemon=True).start()
print('Backup thread started')

model = YOLO('yolo26s.pt')
results = model.train(
    data=data_yaml,
    imgsz=640,
    batch=32,
    epochs=100,
    patience=20,
    fraction=1.0,
    optimizer='MuSGD',
    lr0=0.01,
    weight_decay=0.0005,
    amp=True,
    device='0',
    save_period=10,
    mosaic=1.0,
    mixup=0.05,
    copy_paste=0.5,
    hsv_h=0.02,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=20.0,
    translate=0.15,
    scale=0.8,
    flipud=0.3,
    fliplr=0.5,
    project='/content/runs',
    name='anti_uav_2class_birdvsdrone',
)
stop_backup.set()
print(f'Training complete: {results.save_dir}')


In [ ]:
# Archive and save to Drive
import zipfile, os, shutil
runs_dir = '/content/runs/anti_uav_2class_birdvsdrone'
archive_drive = '/content/drive/MyDrive/anti_uav_2class_birdvsdrone_full.zip'

print('Archiving...')
with zipfile.ZipFile(archive_drive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(runs_dir):
        for file in files:
            filepath = os.path.join(root, file)
            arcname = os.path.relpath(filepath, '/content')
            zf.write(filepath, arcname)
            print(f'  {arcname} ({os.path.getsize(filepath)/1e6:.1f} MB)')

print(f'Saved: {archive_drive} ({os.path.getsize(archive_drive)/1e6:.1f} MB)')
